In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modules/axion.py").exists())
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))

import numpy as np
from pathlib import Path

from pythtb import W90
from pythtb.utils import finite_diff_coeffs, levi_civita
from tensorflow import constant as const, complex64
import tensorflow as tf
import tensorflow.linalg as tfla
import matplotlib.pyplot as plt

In [2]:
BASE = DATA / "MnBi2Te4/base/vdw_U/U_0.0/150300bc889"   # the symmetrized models are in <run>/symmetrized/
mode = DATA / "MnBi2Te4/phonon/T2-/mode1/Q_0p01"

In [3]:
def get_path(symmetrized, mode):
    """Base and mode Wannier folders, symmetrized or raw."""
    base_path = str(BASE / "symmetrized") if symmetrized else str(BASE)
    mode_path = str(mode / "symmetrized") if symmetrized else str(mode)
    missing = [p for p in (Path(base_path, "MnBi2Te4_hr.dat"), Path(mode_path, "MnBi2Te4_hr.dat")) if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing Wannier files:\n" + "\n".join(map(str, missing)))
    return base_path, mode_path

In [4]:
print("Loading Wannier90 models...")

w90s = {}
models = {}

for symm in [True, False]:
    base_path, mode_path = get_path(symm, mode)

    w90_base = W90(base_path, r"MnBi2Te4")
    w90_mode1 = W90(mode_path, r"MnBi2Te4")

    print("Building PythTB models...")
    model_base = w90_base.model(
        zero_energy=0,
        min_hopping_norm=1e-5,
        max_distance=125,
        ignorable_imaginary_part=None,
    )

    model_mode1 = w90_mode1.model(
        zero_energy=0,
        min_hopping_norm=1e-5,
        max_distance=125,
        ignorable_imaginary_part=None,
    )

    models[symm] = {'base': model_base, 'mode1': model_mode1}
    w90s[symm] = {'base': w90_base, 'mode1': w90_mode1}

Loading Wannier90 models...
Found all required symmetrized files.
Building PythTB models...
Found all required symmetrized files.
Building PythTB models...


In [5]:
d_beta = 0.01  # [Angstroms] finite difference step in beta
n_occ = 58

In [6]:
Hams = {}
vels = {}

nk = 4
k_mesh = model_base.k_uniform_mesh([nk, nk, nk], include_endpoints=False)

def vel_fd(H_k, mu, dk_mu, order_eps, mode="central"):
    coeffs, stencil = finite_diff_coeffs(order=order_eps, mode=mode)
    fd_sum = np.zeros_like(H_k)
    for s, c in zip(stencil, coeffs):
        fd_sum += c * np.roll(H_k, shift=-s, axis=mu)
    return fd_sum / dk_mu

for symm in [True, False]:
    print(f"Constructing Hamiltonian and velocity for symmetrized={symm}...")
    model_base = models[symm]['base']
    model_mode1 = models[symm]['mode1']

    # One mode pair: [base (beta=0), mode_1 (beta=1)]
    H = np.zeros((k_mesh.shape[0], 2, model_base.norb, model_base.norb), dtype=np.complex64)
    v_k = np.zeros((3, k_mesh.shape[0], model_base.norb, model_base.norb), dtype=np.complex64)

    print(f"Filling base (beta=0) and {mode} (beta=1)...")
    batch_size = 50
    for i in range(0, len(k_mesh), batch_size):
        print(f"{i} / {len(k_mesh)} k points processed")
        k_batch = k_mesh[i:i + batch_size]

        H_base = model_base.hamiltonian(k_batch, flatten_spin_axis=True)
        H_mode1 = model_mode1.hamiltonian(k_batch, flatten_spin_axis=True)

        v_base = model_base.velocity(k_batch, flatten_spin_axis=True)
        v_mode1 = model_mode1.velocity(k_batch, flatten_spin_axis=True)

        H[i:i + batch_size, 0, :, :] = H_base
        H[i:i + batch_size, 1, :, :] = H_mode1
        v_k[:, i:i + batch_size, :, :] = v_base


    v_beta = (H[:, 1, :, :] - H[:, 0, :, :]) / d_beta
    v = np.concatenate((v_k, v_beta[np.newaxis, :, ...]), axis=0)

    Hams[symm] = H
    vels[symm] = v

Constructing Hamiltonian and velocity for symmetrized=True...
Filling base (beta=0) and mode_1 (beta=1)...
0 / 64 k points processed
50 / 64 k points processed
Constructing Hamiltonian and velocity for symmetrized=False...
Filling base (beta=0) and mode_1 (beta=1)...
0 / 64 k points processed
50 / 64 k points processed


In [7]:
H_sym = Hams[True]
H_nosym = Hams[False]

evals_sym, evecs_sym = np.linalg.eigh(H_sym[:, 0, ...])
evals1_sym, evecs1_sym = np.linalg.eigh(H_sym[:, 1, ...])

evals_nosym, evecs_nosym = np.linalg.eigh(H_nosym[:, 0, ...])
evals1_nosym, evecs1_nosym = np.linalg.eigh(H_nosym[:, 1, ...])
print("Eigvals shape:", evals_sym.shape, "Eigvecs shape:", evecs_sym.shape)

Eigvals shape: (64, 92) Eigvecs shape: (64, 92, 92)


In [8]:
V_occ_sym = evecs_sym[:, :, :n_occ]                     # column i, not row i
P_occ_sym = V_occ_sym @ np.swapaxes(V_occ_sym.conj(), -1, -2)
V1_occ_sym = evecs1_sym[:, :, :n_occ]
P1_occ_sym = V1_occ_sym @ np.swapaxes(V1_occ_sym.conj(), -1, -2)

V_occ_nosym = evecs_nosym[:, :, :n_occ]
P_occ_nosym = V_occ_nosym @ np.swapaxes(V_occ_nosym.conj(), -1, -2)
V1_occ_nosym = evecs1_nosym[:, :, :n_occ]
P1_occ_nosym = V1_occ_nosym @ np.swapaxes(V1_occ_nosym.conj(), -1, -2)

In [10]:
diff_sym = np.amax(np.abs(P_occ_sym - P1_occ_sym))
diff_nosym = np.amax(np.abs(P_occ_nosym - P1_occ_nosym))

print(f"Max |P_occ - P1_occ| for symmetrized: {diff_sym:.2e}")
print(f"Max |P_occ - P1_occ| for non-symmetrized: {diff_nosym:.2e}")

Max |P_occ - P1_occ| for symmetrized: 5.49e-03
Max |P_occ - P1_occ| for non-symmetrized: 2.04e-02


In [48]:
diff_sym = np.amax(np.abs(P_occ_sym - P1_occ_sym))
diff_nosym = np.amax(np.abs(P_occ_nosym - P1_occ_nosym))

print(f"Max |P_occ - P1_occ| for symmetrized: {diff_sym:.2e}")
print(f"Max |P_occ - P1_occ| for non-symmetrized: {diff_nosym:.2e}")

Max |P_occ - P1_occ| for symmetrized: 6.05e-01
Max |P_occ - P1_occ| for non-symmetrized: 2.04e-02


In [49]:
diff_e_sym = np.amax(np.abs(evals_sym - evals1_sym))
diff_e_nosym = np.amax(np.abs(evals_nosym - evals1_nosym))

print(f"Max |evals - evals1| for symmetrized: {diff_e_sym:.2e}")
print(f"Max |evals - evals1| for non-symmetrized: {diff_e_nosym:.2e}")

Max |evals - evals1| for symmetrized: 1.12e-02
Max |evals - evals1| for non-symmetrized: 2.90e-02


In [12]:
diff_H_sym = np.amax(np.abs(H_sym[:, 0, ...] - H_sym[:, 1, ...]))
diff_H_nosym = np.amax(np.abs(H_nosym[:, 0, ...] - H_nosym[:, 1, ...]))
diff_Hbase_symm_nosymm = np.amax(np.abs(H_sym[:, 0, ...] - H_nosym[:, 0, ...]))
diff_Hmode1_symm_nosymm = np.amax(np.abs(H_sym[:, 1, ...] - H_nosym[:, 1, ...]))

print(f"Max |H_base - H_mode1| for symmetrized: {diff_H_sym:.2e}")
print(f"Max |H_base - H_mode1| for non-symmetrized: {diff_H_nosym:.2e}")
print(f"Max |H_base_sym - H_base_nosym|: {diff_Hbase_symm_nosymm:.2e}")
print(f"Max |H_mode1_sym - H_mode1_nosym|: {diff_Hmode1_symm_nosymm:.2e}")

Max |H_base - H_mode1| for symmetrized: 2.00e-02
Max |H_base - H_mode1| for non-symmetrized: 4.03e-02
Max |H_base_sym - H_base_nosym|: 1.75e-01
Max |H_mode1_sym - H_mode1_nosym|: 1.76e-01


In [52]:
diff_H_sym = np.amax(np.abs(H_sym[:, 0, ...] - H_sym[:, 1, ...]))
diff_H_nosym = np.amax(np.abs(H_nosym[:, 0, ...] - H_nosym[:, 1, ...]))
diff_Hbase_symm_nosymm = np.amax(np.abs(H_sym[:, 0, ...] - H_nosym[:, 0, ...]))
diff_Hmode1_symm_nosymm = np.amax(np.abs(H_sym[:, 1, ...] - H_nosym[:, 1, ...]))

print(f"Max |H_base - H_mode1| for symmetrized: {diff_H_sym:.2e}")
print(f"Max |H_base - H_mode1| for non-symmetrized: {diff_H_nosym:.2e}")
print(f"Max |H_base_sym - H_base_nosym|: {diff_Hbase_symm_nosymm:.2e}")
print(f"Max |H_mode1_sym - H_mode1_nosym|: {diff_Hmode1_symm_nosymm:.2e}")

Max |H_base - H_mode1| for symmetrized: 3.52e+00
Max |H_base - H_mode1| for non-symmetrized: 4.03e-02
Max |H_base_sym - H_base_nosym|: 1.75e-01
Max |H_mode1_sym - H_mode1_nosym|: 3.53e+00


In [34]:
model_sym = models[True]['base']
model_nosym = models[False]['base']

In [38]:
model_nosym.orb_vecs

array([[ 2.11035894e-01,  2.10903947e-01,  2.11035899e-01],
       [ 2.09968275e-01,  2.09552298e-01,  2.09968275e-01],
       [ 2.20277009e-01,  1.94250495e-01,  2.20277064e-01],
       [ 2.23292300e-01,  1.86176266e-01,  2.23292314e-01],
       [ 2.02769735e-01,  2.29264512e-01,  2.02769666e-01],
       [ 1.98227308e-01,  2.36304359e-01,  1.98227299e-01],
       [ 7.09990331e-01,  7.09562132e-01,  7.09990340e-01],
       [ 7.11055822e-01,  7.10915484e-01,  7.11055822e-01],
       [ 7.23264844e-01,  6.86273533e-01,  7.23264848e-01],
       [ 7.20251487e-01,  6.94344019e-01,  7.20251450e-01],
       [ 6.98277039e-01,  7.36246812e-01,  6.98277020e-01],
       [ 7.02819666e-01,  7.29206305e-01,  7.02819707e-01],
       [ 2.90006014e-01,  2.90445159e-01,  2.90006010e-01],
       [ 2.88940524e-01,  2.89091810e-01,  2.88940524e-01],
       [ 2.76731490e-01,  3.13733764e-01,  2.76731508e-01],
       [ 2.79744894e-01,  3.05663275e-01,  2.79744862e-01],
       [ 3.01719317e-01,  2.63760490e-01

In [39]:
model_sym.orb_vecs

array([[0.21100048, 0.21100048, 0.21100048],
       [0.20983861, 0.20983861, 0.20983861],
       [0.21379171, 0.20724076, 0.21379171],
       [0.21405484, 0.20467022, 0.21405484],
       [0.20942514, 0.21597611, 0.20942514],
       [0.20779915, 0.2171838 , 0.20779915],
       [0.70983861, 0.70983861, 0.70983861],
       [0.71100048, 0.71100048, 0.71100048],
       [0.71405397, 0.70466935, 0.71405397],
       [0.71379084, 0.70723989, 0.71379084],
       [0.70780001, 0.71718466, 0.70780001],
       [0.709426  , 0.71597698, 0.709426  ],
       [0.29016139, 0.29016139, 0.29016139],
       [0.28899952, 0.28899952, 0.28899952],
       [0.28594429, 0.29532891, 0.28594429],
       [0.28620742, 0.29275837, 0.28620742],
       [0.29220172, 0.28281707, 0.29220172],
       [0.29057573, 0.28402476, 0.29057573],
       [0.78899952, 0.78899952, 0.78899952],
       [0.79016139, 0.79016139, 0.79016139],
       [0.78620655, 0.7927575 , 0.78620655],
       [0.78594342, 0.79532804, 0.78594342],
       [0.